# Notebook 06 — Burning Risk Score (BRS)

**Purpose:** Combine three normalised fire metrics into a composite 0–100 Burning Risk Score (BRS) and classify each district into Low / Moderate / High / Critical risk categories.

**Weights:** fire frequency 40%, fire intensity (FRP) 30%, trend slope 30%  
**Input:** `data/processed/fire_stats.csv`, `data/processed/fire_trends.csv`  
**Output:** `data/processed/burning_risk_scores.csv`

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load inputs 
stats  = pd.read_csv('../Data/Processed/fire_stats.csv')
trends = pd.read_csv('../Data/Processed/fire_trends.csv')

print(f"Stats shape : {stats.shape}")
print(f"Trends shape: {trends.shape}")

Stats shape : (386, 8)
Trends shape: (43, 6)
C:\Users\DSSON\Documents\M.Sc. Sem 2\Residue_Project\Notebooks


In [ ]:
# Aggregate fire stats to one row per district (mean across all years) 
means = (
    stats
    .groupby('district')
    .agg(mean_count=('fire_count', 'mean'),
         mean_frp  =('mean_frp',   'mean'))
    .reset_index()
)
print(means.head())


   district    mean_count  mean_frp
0    Ambala    869.666667  5.003542
1  Amritsar   4014.888889  5.697614
2   Barnala   6303.888889  7.457589
3  Bathinda  10259.777778  7.294171
4   Bhiwani    219.888889  6.180912


In [ ]:
# Merge with trend slope 
# Sen's slope gives the estimated annual increase in fire count.
df = means.merge(trends[['district', 'slope']], on='district', how='inner')

# Clip negative slopes to zero: a decreasing trend contributes zero additional risk.
# This ensures we do not reward improvement — we only penalise growth.
df['slope'] = df['slope'].clip(lower=0)

print(f"Merged shape: {df.shape}")
print(df.head())


Merged shape: (43, 4)
   district    mean_count  mean_frp  slope
0    Ambala    869.666667  5.003542   70.8
1  Amritsar   4014.888889  5.697614  374.8
2   Barnala   6303.888889  7.457589  343.2
3  Bathinda  10259.777778  7.294171  666.8
4   Bhiwani    219.888889  6.180912   12.6


In [ ]:
# Normalise all three features to 0–100 
# MinMaxScaler maps each feature's full range to [0, 100].
scaler = MinMaxScaler(feature_range=(0, 100))
df[['norm_count', 'norm_frp', 'norm_slope']] = scaler.fit_transform(
    df[['mean_count', 'mean_frp', 'slope']]
)

print("Normalisation complete. Sample:")
print(df[['district', 'norm_count', 'norm_frp', 'norm_slope']].head())


Normalisation complete. Sample:
   district  norm_count   norm_frp  norm_slope
0    Ambala    6.455078  30.226639    9.532786
1  Amritsar   30.029315  46.097302   50.464521
2   Barnala   47.185949  86.340934   46.209775
3  Bathinda   76.836337  82.604216   89.780530
4   Bhiwani    1.584830  57.148414    1.696513


In [ ]:
# Compute Burning Risk Score 
# BRS = 0.40 × fire frequency + 0.30 × fire intensity + 0.30 × trend growth
# Weights reflect that frequency (extent) matters most, while intensity
# and trend direction carry equal secondary importance.
df['BRS'] = (
    0.40 * df['norm_count'] +
    0.30 * df['norm_frp']   +
    0.30 * df['norm_slope']
).round(1)

# Classify into risk categories 
def classify(score):
    if   score >= 75: return 'Critical'
    elif score >= 50: return 'High'
    elif score >= 25: return 'Moderate'
    else:             return 'Low'

df['risk_class'] = df['BRS'].apply(classify)

print("Risk class distribution:")
print(df['risk_class'].value_counts().to_string())
print()
print("Top 10 districts by BRS:")
print(
    df[['district', 'BRS', 'risk_class']]
    .sort_values('BRS', ascending=False)
    .head(10)
    .to_string(index=False)
)

Risk class distribution:
risk_class
Low         24
Moderate     8
High         8
Critical     3

Top 10 districts by BRS:
  district  BRS risk_class
   Sangrur 93.1   Critical
 Ferozepur 83.5   Critical
  Bathinda 82.4   Critical
      Moga 69.2       High
   Patiala 67.7       High
   Muktsar 64.0       High
Tarn Taran 60.1       High
     Mansa 58.8       High
   Barnala 58.6       High
  Faridkot 56.8       High


“The Burning Risk Score (BRS) was computed using a weighted linear combination approach, a standard method in GIS-based multi-criteria evaluation (Malczewski, 1999). Weights were assigned based on domain knowledge, where fire frequency was given higher importance (0.40) as it represents spatial extent and recurrence of fire events. Fire intensity (FRP) and temporal trend (Sen’s slope) were assigned equal weights (0.30 each) as secondary indicators of fire severity and temporal dynamics. This weighting scheme is consistent with established risk assessment frameworks such as those proposed by the IPCC, where multiple contributing factors are combined based on their relative importance.”

In [ ]:
# Save final risk scores 
df.to_csv('../Data/Processed/burning_risk_scores.csv', index=False)
print("Saved: Data/Processed/burning_risk_scores.csv")
print(f"   Rows    : {len(df)}")
print(f"   Columns : {df.columns.tolist()}")

✅ Saved: Data/Processed/burning_risk_scores.csv
   Rows    : 43
   Columns : ['district', 'mean_count', 'mean_frp', 'slope', 'norm_count', 'norm_frp', 'norm_slope', 'BRS', 'risk_class']
